# 08 — Modèle de bruit Hanoï

**Entraînement direct sur nos mesures + validation honnête.**

Ce notebook :
1. calcule les features de morphologie OSM (rayon 300 m) pour nos points de mesure ;
2. lance l'évaluation complète — **blocs spatiaux de 600 m, baselines, ablation, IC bootstrap** —
   via `scripts/04_evaluate_models.py` ;
3. entraîne le modèle final sur tous les points et le sauvegarde ;
4. renvoie vers `scripts/07_export_gama_inputs.py` pour produire la carte.

### Ce qui a changé (correction méthodologique, août 2026)

- **La CV groupée sur `lat/lon.round(3)` (~110 m) est supprimée.** Les features sont des
  agrégats sur un disque de **rayon 300 m** : deux points distants de 110 m partagent plus de
  85 % de leur disque. Le modèle voyait des quasi-jumeaux de ses points de test, et le
  R² 0.45 annoncé n'était pas hors-échantillon (Roberts et al. 2017, *Ecography*).
  Protocoles retenus : **block-CV 600 m**, **buffered leave-one-out (300 m)**, **leave-one-site-out**.
- **La grille « Bach Khoa » est supprimée.** Elle couvrait un quartier **sans aucune mesure**,
  prédit par un modèle dont le leave-one-site-out est négatif. La carte est désormais produite
  par `scripts/07_export_gama_inputs.py`, **sur l'emprise réellement échantillonnée** (3 sites + 400 m).
- **Des baselines et une ablation sont ajoutées** pour mesurer l'apport propre de la morphologie.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
# feature and path code now lives in the installed package (pip install -e .)

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd

CRS_HANOI = 'EPSG:32648'   # UTM 48N
R = 300                    # rayon des features de morphologie (m)
AREA_M2 = np.pi * R ** 2
MEASURES = '../data/processed/measurements.csv'
PROC = '../data/interim'
os.makedirs(PROC, exist_ok=True)
os.makedirs('../models', exist_ok=True)

hanoi = pd.read_csv(MEASURES, parse_dates=['timestamp'])
hanoi['hour'] = hanoi.timestamp.dt.hour
hanoi['is_weekend'] = hanoi.timestamp.dt.dayofweek.isin([5, 6]).astype(int)
hanoi = hanoi.dropna(subset=['noise_dB', 'latitude', 'longitude']).reset_index(drop=True)
print(f'{len(hanoi)} mesures · sites {hanoi.site.value_counts().to_dict()}')
print(f'dB {hanoi.noise_dB.min():.0f}-{hanoi.noise_dB.max():.0f} · sd {hanoi.noise_dB.std():.1f}')

## 1. Cache OSM autour des points de mesure

Téléchargé une seule fois puis relu. C'est ce cache que consomment ensuite
`evaluate_models.py` et `export_gama_zones.py` — une seule source de vérité géographique.

In [ ]:
HB_PATH = f'{PROC}/hanoi_sites_buildings.gpkg'
HG_PATH = f'{PROC}/hanoi_sites_roads.graphml'
MARGIN = 0.006   # ~600 m, pour que les disques de 300 m des points de bord soient complets

if not (os.path.exists(HB_PATH) and os.path.exists(HG_PATH)):
    print('Téléchargement OSM autour des points de mesure (long la première fois)...')
    ox.settings.timeout = 600
    bbox = (hanoi.longitude.min() - MARGIN, hanoi.latitude.min() - MARGIN,
            hanoi.longitude.max() + MARGIN, hanoi.latitude.max() + MARGIN)
    hb = ox.features_from_bbox(bbox, tags={'building': True})
    hb = hb[hb.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
    hb[['geometry']].to_file(HB_PATH, driver='GPKG')
    ox.save_graphml(ox.graph_from_bbox(bbox, network_type='drive'), HG_PATH)

from noise_hanoi import features as egz     # load_osm() / morphology() : logique partagée, pas dupliquée
_, bld_c, nodes, edges = egz.load_osm()
print(f'{len(bld_c)} bâtiments · {len(edges)} segments routiers en cache')

## 2. Features de morphologie (rayon 300 m) pour nos points

In [ ]:
pts = gpd.GeoDataFrame(hanoi, geometry=gpd.points_from_xy(hanoi.longitude, hanoi.latitude),
                       crs='EPSG:4326').to_crs(CRS_HANOI)
feats = egz.morphology(pts, bld_c, nodes, edges)

MORPHO   = ['built_area_ratio', 'road_density_km_km2', 'intersection_count', 'dist_road_m']
TIME     = ['hour', 'is_weekend']
FEATURES = MORPHO + TIME

df = pd.concat([hanoi, feats], axis=1)
df['x'], df['y'] = pts.geometry.x.values, pts.geometry.y.values
# Bloc spatial de 600 m = 2 x le rayon des features : l'unité de découpage de la CV.
df['block'] = (np.floor(df.x / 600).astype(int).astype(str) + '_' +
               np.floor(df.y / 600).astype(int).astype(str))
print(f'{df.block.nunique()} blocs spatiaux de 600 m pour {len(df)} points '
      f'(médiane {df.groupby("block").size().median():.0f} points/bloc)')
df[FEATURES].describe().round(2)

## 3. Évaluation honnête — baselines, ablation, intervalles de confiance

Tout est dans `scripts/04_evaluate_models.py` (source unique, réutilisable hors notebook).
Il évalue **huit modèles sur exactement les mêmes découpages** :

| | |
|---|---|
| `global_mean`, `site_mean` | planchers |
| `site_hour_mean` | table de correspondance (site, heure) — **le baseline à battre** |
| `dist_road`, `idw` | physique minimale, interpolation pure |
| `lgbm_time`, `lgbm_morpho` | ablations |
| `lgbm_full` | le modèle du projet |

La ligne à lire est **`morphology_gain`** : ΔR² et ΔMAE de `lgbm_full` face à `site_hour_mean`.
C'est la réponse chiffrée à « la morphologie urbaine apporte-t-elle quelque chose ? ».

In [ ]:
!cd .. && python3 scripts/04_evaluate_models.py

In [ ]:
import json
metrics = json.load(open('../models/metrics.json'))

ref = metrics['meta']['headline_protocol']          # 'bloo' si disponible, sinon 'block_cv'
rows = [{'modèle': m['label'], 'R²': round(m['r2'], 3),
         'IC95 R²': f"[{m['r2_ci95'][0]:.2f}, {m['r2_ci95'][1]:.2f}]",
         'MAE': round(m['mae'], 2), 'r': round(m['r'], 2)}
        for m in metrics[ref]['models'].values()]
print(f"Protocole de référence : {metrics[ref]['label']}")
display(pd.DataFrame(rows))

g = metrics[ref]['morphology_gain']
print(f"\nApport propre de la morphologie : ΔR² {g['delta_r2']:+.3f} · ΔMAE {g['delta_mae_dB']:+.2f} dB")
print('\nLeave-one-site-out (généralisation à une typologie non vue) :')
for s, v in metrics['loso_per_site'].items():
    print(f"  {s:18} n={v['n']:3}  R² {v['r2']:6.2f}  MAE {v['mae']:5.2f}")

## 4. Comparaison : transfert Ouganda → Hanoï

Gardé comme **résultat méthodologique** (voir `docs/negative-results.md`), pas comme
méthode. Évalué sur les mêmes blocs de 600 m que le reste, pour que la comparaison soit juste.

In [ ]:
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr

uganda = joblib.load('../models/surrogate_lgbm_v2_uganda.pkl')

# Le modèle Ouganda est appliqué tel quel, avec un offset ajusté sur les seuls blocs
# d'apprentissage de chaque pli — sinon l'offset lui-même fuiterait.
oof = np.full(len(df), np.nan)
for tr, te in GroupKFold(5).split(df, df.noise_dB, df.block):
    off = (df.noise_dB.iloc[tr] - uganda.predict(df[FEATURES].iloc[tr])).mean()
    oof[te] = uganda.predict(df[FEATURES].iloc[te]) + off

print(f'Transfert Ouganda + offset (blocs 600 m) : '
      f'r {pearsonr(df.noise_dB, oof)[0]:.2f} | R² {r2_score(df.noise_dB, oof):.2f} | '
      f'MAE {mean_absolute_error(df.noise_dB, oof):.1f} dB')
print(f"À comparer au modèle direct : R² {metrics[ref]['models']['lgbm_full']['r2']:.2f}")

## 5. Modèle final et carte

Le modèle final est entraîné sur **tous** les points (aucune métrique n'en est tirée : les
chiffres publiés viennent tous de la section 3).

**La carte n'est plus produite ici.** `scripts/07_export_gama_inputs.py` la génère sur l'emprise
réellement échantillonnée — les 3 sites + 400 m de marge — avec une colonne par heure (h5…h21).
Prédire au-delà de cette emprise reviendrait à extrapoler vers des typologies que le
leave-one-site-out montre non maîtrisées.

In [ ]:
import lightgbm as lgb

final = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15,
                          min_child_samples=10, random_state=0, verbose=-1)
final.fit(df[FEATURES], df.noise_dB)
final.booster_.save_model('../models/surrogate_lgbm_hanoi_direct.txt')

imp = pd.Series(final.booster_.feature_importance('gain'), index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 3.2), title='Importance des features (gain) — modèle final')

print('Modèle sauvegardé -> outputs/models/surrogate_lgbm_hanoi_direct.txt')
print('\nSuite du pipeline :')
print('  python3 scripts/07_export_gama_inputs.py    # carte sur les 3 zones mesurées, 5h-21h')
print('  python3 scripts/validate_simulation.py  # confrontation grille <-> mesures (in-sample)')
print('  python3 scripts/build_report.py         # rapport PDF (lit metrics.json)')